# 05 - Final Export and Compliance

Run after training and evaluation. Export selected weights, construct a separate candidate, and check its artifacts. The starter agent is preserved. Nothing is uploaded automatically.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project and Dependencies

Keep Colab's existing CUDA PyTorch. Restart only if pip explicitly requires it.

In [ ]:
from pathlib import Path
import sys
import subprocess

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl").is_dir():
    raise FileNotFoundError(f"Project files are missing from {PROJECT_ROOT}. See README.md.")
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", str(PROJECT_ROOT / "requirements_colab.txt")])

## Run Configuration

Edit configs/default.yaml once for the workflow. Use a new run_id for a changed experiment.

In [ ]:
from chess_rl.config import load_config, prepare_directories
from chess_rl.reproducibility import metadata, read_json, atomic_json, sha256

prepare_directories(PROJECT_ROOT)
cfg = load_config(PROJECT_ROOT)
print("Run:", cfg["run_id"])
print("Runtime:", metadata())

## Require Frozen Selection

Earlier workflow completion is required. Failure does not trigger retraining.

In [ ]:
from chess_rl.export import freeze_requirement
selection, checkpoint = freeze_requirement(PROJECT_ROOT, cfg["run_id"])
print("Frozen checkpoint:", checkpoint)
print("Hash:", selection["checkpoint_sha256"])

## Optional ONNX Dependencies

PyTorch state_dict is the default. ONNX is final-only and has separate runtime thread settings.

In [ ]:
if selection["config"]["export"]["format"] == "onnx":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "onnx", "onnxruntime"])

## Export, Inspect, and Play

Build the ZIP, check CPU reconstruction, measure batch-one latency, and play real games. The 5 ms target is experimental. Unreplicated isolation and human provenance review are reported as not verified.

In [ ]:
from chess_rl.export import final_checks
report = final_checks(PROJECT_ROOT, cfg["run_id"], live_games=16)
print("Status:", report["status"])
print("Archive:", report.get("archive"))
print("Uncompressed bytes:", report.get("unzipped_bytes"))
print("CPU inference milliseconds:", report.get("inference_ms"))
print("Checks:", report["checks"])

## Inspect Files and Logs

The candidate contains only its runtime source, needed weights, configuration, and manifest.

In [ ]:
print("Candidate directory:", PROJECT_ROOT / "submission_candidate")
print("Report:", PROJECT_ROOT / "results" / cfg["run_id"] / "final_compliance.json")
print("Live results:", report.get("live_results"))
print("Original starter repository was not replaced.")

## Platform Submission

The reported ZIP can be uploaded on the dashboard. Only platform validation establishes acceptance. Inspect a failed final report; this workflow does not automatically rerun training.